# Install openai-agents SDK

In [1]:
!pip install -Uq openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.5/128.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 7.6 MB/s eta 0:00:00


# Make your Jupyter Notebook capable of running asynchronous functions.

In [4]:
import nest_asyncio
nest_asyncio.apply()

# Run Google Gemini with OPENAI-Agent SDK

In [8]:
import os
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
from agents.run import RunConfig


API_KEY = os.environ.get("AIHUBMIX_API_KEY")
BASE_URL = os.environ.get("AIHUBMIX_BASE_URL")

# Check if the API key is present; if not, raise an error
if not API_KEY:
    raise ValueError("API_KEY is not set. Please ensure it is defined in your .env file.")

#Reference: https://ai.google.dev/gemini-api/docs/openai
external_client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

model = OpenAIChatCompletionsModel(
    model="gpt-4o",
    openai_client=external_client
)

config = RunConfig(
    model=model,
    model_provider=external_client,
    tracing_disabled=True
)

# Streaming Text code

In [9]:
import asyncio

from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner


async def main():
    agent = Agent(
        name="Joker",
        instructions="You are a helpful assistant. 只输出中文。",
        model=model,
    )

    result = Runner.run_streamed(agent, input="Please tell me 5 jokes.", run_config=config)
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)



asyncio.run(main())

1. 为什么数学书总是很忧伤？因为里面有太多的问题。

2. 一只鸭子走到药店，对药剂师说：“给我一管唇膏。”药剂师问：“要刷卡吗？”鸭子回答：“不，直接涂在嘴上就行。”

3. 小明问老师：“一加一为什么等于二？”老师回答：“因为数学没商量。”

4. 兔子去点心店买胡萝卜，店员说：“不好意思，我们只卖蛋糕。”兔子说：“没关系，我带自己的胡萝卜来吃。”

5. 为什么大象不喜欢玩捉迷藏？因为它的鼻子总是露出来。

# Stream item code

In [10]:
import asyncio
import random

from agents import Agent, ItemHelpers, Runner, function_tool


@function_tool
def how_many_jokes() -> int:
    return random.randint(1, 10)


async def main():
    agent = Agent(
        name="Joker",
        instructions="First call the `how_many_jokes` tool, then tell that many jokes.",
        tools=[how_many_jokes],
        model=model,
    )

    result = Runner.run_streamed(
        agent,
        input="Hello",
        run_config=config

    )
    print("=== Run starting ===")
    async for event in result.stream_events():
        # We'll ignore the raw responses event deltas
        if event.type == "raw_response_event":
            continue
        elif event.type == "agent_updated_stream_event":
            print(f"Agent updated: {event.new_agent.name}")
            continue
        elif event.type == "run_item_stream_event":
            if event.item.type == "tool_call_item":
                print("-- Tool was called")
            elif event.item.type == "tool_call_output_item":
                print(f"-- Tool output: {event.item.output}")
            elif event.item.type == "message_output_item":
                print(f"-- Message output:\n {ItemHelpers.text_message_output(event.item)}")
            else:
                pass  # Ignore other event types




try:
  asyncio.run(main())
except:
  pass
print("=== Run complete ===")

=== Run starting ===
Agent updated: Joker
-- Message output:
 Hi there! How can I assist you today?
=== Run complete ===
